# Notebook 48 — Option 6: pool every real simulation this project has already paid for

## Purpose

Your own idea (2026-09-06): rather than committing to one "clean" training prior, pool
**every currently-valid real simulation already generated** across every prior tried
this project (lognormal, alpha-only-mixed, and any future banks from notebooks
44/47), and train one SNPE posterior on the union — "even if we get there by using a
ridiculous amount of simulations." **You are running this notebook, not Claude.**

## Why this is mathematically valid, not just a hack

`sbi`'s `SNPE_C` supports exactly this via its atomic-loss "proposal correction"
machinery (verified directly against the installed `sbi==0.26.1` source before writing
this notebook, not assumed from memory): each call to `append_simulations(theta, x,
proposal=P)` tags that batch with **its own** originating distribution `P`. Calling
`append_simulations` multiple times — once per historical bank, each with *that bank's
own, exactly-known prior* as `proposal=` — then a **single** final `.train()` call,
trains one flow on the union of all of them, correctly reweighted via the atomic loss
toward whatever prior you declare at `SNPE(prior=...)` construction (here:
`box_uniform_5d()`, the full parameter box, the most general choice). This is the
textbook multi-round mechanism, just applied retroactively to pool independent sources
rather than to sequential refinement rounds. **A single final `.train()` call also
sidesteps the early-stopping bug found in this session's multi-round pilot** (that bug
only bit repeated `.train()` calls; here there's only one).

## What's actually in the pool right now (checked against disk before writing this)

Per `HANDOFF.md`'s repeated warning, **only `_stage1bfix`-suffixed (or otherwise
confirmed post-both-fixes) banks are safe to use** — everything else either predates
the Item-6 physics bug fix (a different governing ODE) or the Stage-1b warm-start fix
(non-convergent steady states silently accepted). Checked directly:

- `wu2003_sbi_train_sc2_lognormal_true_n1000_stage1bfix.npz` (n=1000,
  `truncated_lognormal_5d()`) and `wu2003_sbi_train_sc_lognormal_true_n1000_stage1bfix.npz`
  (n=1000, same prior, 72-D not 74-D) turned out to be **the same 1000 theta rows**
  (confirmed via direct array comparison — the "sc2" file is the S-C.2-augmented version
  of the "sc" one) — **only the S-C.2 (74-D) version is included below, to avoid
  double-counting.**
- `wu2003_sbi_train_sc2_true_n1000_mixedprior_stage1bfix.npz` (n=1000,
  `mixed_alpha_uniform_prior()`) — genuinely distinct theta rows (different prior).
- The `_n120`/`_n400` SBC-set files for both priors are also real, valid,
  post-both-fixes simulated data. The `_n120` ("screen") sets are folded into training
  too (more free data); **the `_n400` ("confirm") sets are reserved, NOT pooled into
  training**, so there's still an honest held-out SBC check per source prior.
- All S-B-structure banks (66-D, no feed analyzer) are **excluded** — different
  observation space entirely, not reusable for the S-C.2 model.
- Every other bank in `data/` lacks the `_stage1bfix` suffix and is treated as stale
  (not pooled), per `HANDOFF.md`'s standing warning.

**This gives ~2240 pooled training rows today** (1000+1000 banks + 120+120 screen
sets) — a real, if modest, multiple of any single bank. **The registry below is
designed to be extended**: once `notebooks/44`'s wide-prior bank or `notebooks/47`'s
restricted-full-uniform bank exist, add them (see the commented-out entries) to grow
the pool further, which is presumably closer to the "ridiculous amount" you have in
mind. Re-run this notebook after generating more banks to keep growing the pool.

**One deliberately-excluded source, flagged so it isn't forgotten**: `notebooks/47`'s
bootstrap/bank data (if generated) is drawn from a `RestrictedPrior`, not a plain
`torch.distributions.Distribution` — pooling it correctly requires also loading the
saved classifier object, which adds real complexity. Not attempted here; a natural
follow-up once notebook 47 has results.


In [ ]:
import pathlib, pickle, sys, time

import numpy as np
import pandas as pd
import torch
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
from cstr_sbi.recycle.priors import box_uniform_5d, truncated_lognormal_5d, mixed_alpha_uniform_prior
from cstr_sbi.recycle.physics import PARAM_NAMES
from cstr_sbi.recycle.scenarios import list_closed_loop, FAULT_UNITS
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SC2

DATA = ROOT / "data"
RESULTS = ROOT / "results"
SBI_LOGS = ROOT / "sbi-logs"

STRUCTURE = "S-C.2"
ARCH, HIDDEN, TRANSFORMS = "zuko_nsf", 60, 3
MAX_EPOCHS, STOP_AFTER_EPOCHS = 120, 12
SEED = 0
THRESH, Z0_NOM = 0.85, 0.90
N_REPS, N_POST = 30, 200

DECLARED_PRIOR = box_uniform_5d()  # the widest sensible choice: the full parameter box.


## Step 1 — the bank registry (extend this list as more banks become available)

Each entry: `path` (must exist to be used -- missing entries are skipped, not errors),
`prior_fn` (a zero-arg callable returning the EXACT prior distribution that bank was
drawn from -- this is what gets passed as `proposal=`), `role` (`"train"` folds it into
the pooled training set; `"eval_holdout"` reserves it purely for an independent SBC
check, never trained on).


In [ ]:
BANK_REGISTRY = [
    {"path": DATA / "wu2003_sbi_train_sc2_lognormal_true_n1000_stage1bfix.npz",
     "prior_fn": truncated_lognormal_5d, "role": "train", "label": "lognormal_train_n1000"},
    {"path": DATA / "wu2003_sbi_train_sc2_true_n1000_mixedprior_stage1bfix.npz",
     "prior_fn": mixed_alpha_uniform_prior, "role": "train", "label": "alpha_mixed_train_n1000"},
    {"path": RESULTS / "24a_sysII_sc2_lognormal_sbc_set_n120_stage1bfix.npz",
     "prior_fn": truncated_lognormal_5d, "role": "train", "label": "lognormal_screen_n120_as_extra_train"},
    {"path": RESULTS / "sc2_sbc_set_n120_mixedprior_stage1bfix.npz",
     "prior_fn": mixed_alpha_uniform_prior, "role": "train", "label": "alpha_mixed_screen_n120_as_extra_train"},
    {"path": RESULTS / "24a_sysII_sc2_lognormal_sbc_set_n400_stage1bfix.npz",
     "prior_fn": truncated_lognormal_5d, "role": "eval_holdout", "label": "lognormal_confirm_n400_HELDOUT"},
    {"path": RESULTS / "sc2_sbc_set_n400_mixedprior_stage1bfix.npz",
     "prior_fn": mixed_alpha_uniform_prior, "role": "eval_holdout", "label": "alpha_mixed_confirm_n400_HELDOUT"},
    # --- extend here once more banks exist, e.g.: ---
    # {"path": DATA / "wu2003_sbi_train_sc2_true_n1000_wideprior_stage1bfix.npz",
    #  "prior_fn": mixed_wide_prior, "role": "train", "label": "wideprior_train_n1000"},
    # (import mixed_wide_prior from cstr_sbi.recycle.priors if you add this)
]

def load_bank(entry):
    if not entry["path"].exists():
        print(f"  SKIP (not found): {entry['label']} -- {entry['path'].name}")
        return None
    d = np.load(entry["path"])
    theta = np.asarray(d["thetas"], dtype=np.float32)
    summary = np.asarray(d["summaries"], dtype=np.float32)
    assert summary.shape[1] == N_SUMMARIES_SC2, f"{entry['label']}: expected {N_SUMMARIES_SC2}-D summaries, got {summary.shape[1]}"
    print(f"  FOUND: {entry['label']:40s} n={theta.shape[0]:5d}  role={entry['role']}")
    return theta, summary

print("Scanning bank registry:")
loaded = []
for entry in BANK_REGISTRY:
    result = load_bank(entry)
    if result is not None:
        theta, summary = result
        loaded.append({**entry, "theta": theta, "summary": summary})

n_train_total = sum(e["theta"].shape[0] for e in loaded if e["role"] == "train")
n_holdout_total = sum(e["theta"].shape[0] for e in loaded if e["role"] == "eval_holdout")
print(f"\nTotal pooled TRAINING rows: {n_train_total}")
print(f"Total reserved HOLD-OUT rows (not trained on): {n_holdout_total}")


## Step 2 — pool everything into one SNPE object, single final `.train()` call

Each `append_simulations` call tags its batch with *that source's own* prior object as
`proposal=` -- this is what makes the pooling mathematically correct rather than a
naive "pretend it's all IID" hack.


In [ ]:
torch.manual_seed(SEED)
inference = SNPE(prior=DECLARED_PRIOR, density_estimator=posterior_nn(
    model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS, z_score_x="none"))

train_sources = [e for e in loaded if e["role"] == "train"]
for e in train_sources:
    source_prior = e["prior_fn"]()
    inference.append_simulations(torch.tensor(e["theta"]), torch.tensor(e["summary"]), proposal=source_prior)
    print(f"  appended {e['label']}: n={e['theta'].shape[0]}, proposal={e['prior_fn'].__name__}")

t0 = time.time()
est = inference.train(max_num_epochs=MAX_EPOCHS, validation_fraction=0.1,
                       stop_after_epochs=STOP_AFTER_EPOCHS, show_train_summary=False)
posterior = inference.build_posterior(est)
print(f"\ntrained on {n_train_total} pooled rows in {time.time()-t0:.0f}s")

posterior_path = SBI_LOGS / "wu2003_posterior_sc2_pooled_multiproposal_stage1bfix.pkl"
with open(posterior_path, "wb") as f:
    pickle.dump({"posterior": posterior, "structure": "S-C.2", "n_train": n_train_total,
                 "torch_seed": SEED, "z_score_x": "none", "prior": "box_uniform_5d (pooled multi-proposal)",
                 "sources": [e["label"] for e in train_sources]}, f)
print(f"saved {posterior_path}")


## Step 3 — SBC on BOTH reserved held-out sets independently

Same dual-independent-SBC-set convention used everywhere else this project
(`finding_sbc_power_eta_col`: don't trust a single small-N pass). Each held-out set is
evaluated on its own terms (rank statistics computed against that set's own true
thetas) -- this checks calibration in the region each respective prior actually covers,
not the full box (no current post-fix SBC set spans the full box uniformly).


In [ ]:
def run_sbc(posterior, theta_set, summary_set, n_post, seed):
    torch.manual_seed(seed)
    ranks = {name: [] for name in PARAM_NAMES}
    for th, x_obs in zip(theta_set, summary_set):
        with torch.no_grad():
            samp = posterior.sample((n_post,), x=torch.tensor(x_obs, dtype=torch.float32),
                                     show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        for k, name in enumerate(PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
    out = {}
    for name in PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        out[name] = float(ks.pvalue)
    out["min_ks_pvalue"] = min(out[name] for name in PARAM_NAMES)
    return out


holdout_sources = [e for e in loaded if e["role"] == "eval_holdout"]
sbc_results = {}
for e in holdout_sources:
    n_post = min(400, e["theta"].shape[0])
    res = run_sbc(posterior, e["theta"], e["summary"], n_post=n_post, seed=600001)
    sbc_results[e["label"]] = res
    print(f"\nSBC vs {e['label']} (N={e['theta'].shape[0]}):")
    for name in PARAM_NAMES:
        p = res[name]
        print(f"  {name:10s} ks_pvalue={p:.4g}  {'PASS' if p > 0.05 else 'fail'}")
    print(f"  min_ks_pvalue={res['min_ks_pvalue']:.4g}  "
          f"{'*** BOTH-CRITERIA-INDEPENDENT PASS ***' if res['min_ks_pvalue'] > 0.05 else ''}")

pd.DataFrame(sbc_results).to_csv(RESULTS / "48_pooled_multiproposal_sbc_results.csv")


## Step 4 — classification on the real 14-scenario Wu2003 catalogue

The test that actually matters, same protocol as every other classification script
this session.


In [ ]:
def sample_fault_unit(theta_row, thresh=THRESH, z0_nom=Z0_NOM):
    alpha, beta_r, eta_col, xi_reb, z_A0 = theta_row
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    feed_bad = z_A0 < thresh * z0_nom
    n_bad = int(reactor_bad) + int(column_bad) + int(feed_bad)
    if n_bad == 0:
        return "healthy"
    if n_bad >= 2:
        return "multi"
    if reactor_bad:
        return "reactor"
    if column_bad:
        return "column"
    return "feed"


def classify_fault_recycle(samples):
    labels = [sample_fault_unit(row) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS}
    return max(probs, key=probs.__getitem__), probs


def compute_classification_metrics(preds, trues, classes=FAULT_UNITS):
    n = len(classes)
    cm = np.zeros((n, n), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1_per_class = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1_per_class[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    macro_f1 = float(np.mean(list(f1_per_class.values())))
    accuracy = float(np.diag(cm).sum() / cm.sum()) if cm.sum() > 0 else 0.0
    return {"confusion_matrix": cm, "class_names": classes, "per_class_f1": f1_per_class,
            "macro_f1": macro_f1, "accuracy": accuracy}


y0_sb = nominal_warm_start("S-B")
rows = []
for sc in list_closed_loop():
    theta_np = np.asarray(sc.theta(), dtype=np.float32)
    true_class = sc.fault_unit()
    t_h, raw_obs = deterministic_window(sc, structure="S-B", y0=y0_sb)
    t_h_arr = np.asarray(t_h)
    rng = np.random.default_rng(20260705)
    reps = noisy_replicates(raw_obs, n_replicates=N_REPS, rng=rng, noise_pct=0.003)
    for i in range(N_REPS):
        s = compute_summaries(reps[i], STRUCTURE, t_h_arr, theta=theta_np, rng=rng)
        with torch.no_grad():
            samp = posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                     show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        pred, probs = classify_fault_recycle(samp)
        rows.append({"scenario": sc.name, "replicate": i, "true_class": true_class, "pred_class": pred, **probs})
    print(f"  {sc.name:22s} true={true_class}")

eval_df = pd.DataFrame(rows)
eval_df.to_csv(RESULTS / "48_pooled_multiproposal_classification_per_replicate.csv", index=False)
metrics = compute_classification_metrics(eval_df["pred_class"].tolist(), eval_df["true_class"].tolist())
print(f"\n=== Pooled multi-proposal posterior classification ===")
print(f"macro-F1={metrics['macro_f1']:.3f}  accuracy={metrics['accuracy']:.3f}")
print("per-class F1:", {k: round(v, 3) for k, v in metrics['per_class_f1'].items()})
np.savez(RESULTS / "48_pooled_multiproposal_classification_metrics.npz",
          cm=metrics["confusion_matrix"], macro_f1=metrics["macro_f1"], accuracy=metrics["accuracy"],
          class_names=FAULT_UNITS)


## Step 5 — context: compare against every other number this project has produced

| Method | pooled training rows | macro-F1 |
|---|---|---|
| full-uniform-prior SBI (nb31, pre-bugfix, historical) | ~15000 (single prior) | 0.694 |
| narrow-lognormal SBI (S-C.2 baseline) | 1000 (single prior) | 0.07-0.08 |
| alpha-only mixed-prior SBI | 1000 (single prior) | 0.11-0.112 |
| **pooled multi-proposal (this notebook)** | *(see Step 1 tally above)* | *(see Step 4 output above)* |

**If this scores well above 0.11 without needing a genuinely new prior**, that's a
strong, cheap win: it means the bottleneck was never any single prior's own coverage,
but simply *not having pooled the coverage we'd already collected across attempts* —
a very different, more optimistic diagnosis than anything found so far. **If it scores
similarly poorly despite the larger pool**, that's also informative: it would suggest
that merely having *more* data (even spanning multiple priors) doesn't fix the
underlying identifiability/architecture problem, and the coverage gap specifically
*at the real fault-scenario operating points* (not just "more data anywhere") is what
matters — reinforcing the case for options 1/3 (deliberately covering the fault region)
over blind data accumulation.

**Report back**: the Step 1 pooled-row tally, both SBC results from Step 3, and the
classification numbers from Step 4 — all three matter for interpreting this.


## Final acceptance criteria

| # | Criterion | Threshold | Rationale |
|---|---|---|---|
| 1 | Both held-out SBC sets calibrated | min_ks_pvalue > 0.05 for EACH reserved held-out set | the pooled posterior should remain calibrated against both source priors' own independent held-out data, not just one |
| 2 | Classification macro-F1 vs. best-so-far | ≥ 0.11 | does pooling justify its added complexity over any single-prior approach already tried? |
| 3 | Classification macro-F1 (stretch) | ≥ 0.35 | roughly half the historical full-uniform-prior ceiling (0.694) |


In [ ]:
per_holdout_min_ks = {label: res["min_ks_pvalue"] for label, res in sbc_results.items()}
all_holdouts_pass = all(p > 0.05 for p in per_holdout_min_ks.values())
macro_f1 = metrics["macro_f1"]

criteria = [
    {"criterion": "1. ALL held-out SBC sets have min_ks_pvalue > 0.05", "value": min(per_holdout_min_ks.values()) if per_holdout_min_ks else float("nan"),
     "threshold": 0.05, "pass": all_holdouts_pass},
    {"criterion": "2. Classification macro-F1 >= 0.11", "value": macro_f1,
     "threshold": 0.11, "pass": macro_f1 >= 0.11},
    {"criterion": "3. Classification macro-F1 >= 0.35 (stretch)", "value": macro_f1,
     "threshold": 0.35, "pass": macro_f1 >= 0.35},
]

print("=" * 78)
print(f"{'Criterion':<50s} {'Value':>9s} {'Threshold':>10s} {'Result':>8s}")
print("-" * 78)
for c in criteria:
    val_str = f"{c['value']:.3f}" if c["value"] == c["value"] else "N/A"
    print(f"{c['criterion']:<50s} {val_str:>9s} {c['threshold']:>10.2f} {'PASS' if c['pass'] else 'FAIL':>8s}")
print("=" * 78)
print("\nPer-held-out-set min_ks_pvalue:")
for label, p in per_holdout_min_ks.items():
    print(f"  {label:40s} {p:.4g}  {'PASS' if p > 0.05 else 'fail'}")

if criteria[0]["pass"] and criteria[2]["pass"]:
    print("\nVERDICT: PASS (strong) -- pooling sunk simulation cost achieves both "
          "calibration and strong classification. Extend the registry with notebooks "
          "44/47's banks (once they exist) to grow the pool further and re-check.")
elif criteria[0]["pass"] and criteria[1]["pass"]:
    print("\nVERDICT: PASS (modest) -- calibrated and beats the best single-prior result, "
          "but not yet a strong absolute result. Worth extending the pool (more sources) "
          "before drawing a final conclusion.")
elif criteria[1]["pass"]:
    print("\nVERDICT: PARTIAL -- classification improved but calibration did not fully "
          "hold across both held-out sets -- report both numbers, don't claim full "
          "calibration.")
else:
    print("\nVERDICT: FAIL -- merely pooling more data (even spanning multiple priors) did "
          "not fix the underlying problem. This argues for options that deliberately cover "
          "the fault-relevant region (1/3) over blind data accumulation.")
